In [1]:
# This script merges song features with staple scores and fills missing values. This will be used for playlist role annotation.
# Originally, we were going to use these for clustering analysis, but we found that the staple scores were too sparse and numerous, introducing too many dimensions.
import pandas as pd
import sqlite3

conn = sqlite3.connect('dbs/song_features.db')
song_features_df = pd.read_sql_query("SELECT * FROM song_features", conn)
conn.close()

staple_scores_df = pd.read_pickle('dbs/staple_scores.pkl')
staple_scores_df = staple_scores_df.reset_index().rename(columns={'index': 'track_uri'})

merged_df = pd.merge(
    song_features_df,
    staple_scores_df,
    how='left',
    on='track_uri'
)

theme_columns = staple_scores_df.columns.difference(['track_uri'])
merged_df[theme_columns] = merged_df[theme_columns].fillna(0)

In [2]:
# Role annotation
cluster_labels = {
    0: 'Generic',
    1: 'Niche',
    2: 'Mainstream',
    3: 'Curated'}

theme_names_list = staple_scores_df.columns.tolist()
staple_score_columns = [col for col in merged_df.columns if col in theme_names_list]
merged_df[staple_score_columns] = merged_df[staple_score_columns].apply(pd.to_numeric, errors='coerce')
merged_df['role_annotation'] = merged_df['cluster'].map(cluster_labels) + ' ' + merged_df[staple_score_columns].idxmax(axis=1).str.title() # cluster label + highest staple score theme

missing_theme = (merged_df[staple_score_columns].max(axis=1) == 0) | (merged_df[staple_score_columns].isna().all(axis=1)) # Identify rows with no staple scores
merged_df.loc[missing_theme, 'role_annotation'] = merged_df.loc[missing_theme, 'cluster'].map(cluster_labels) # Assign generic role (no theme label) for missing staple scores

In [ ]:
# Save the merged DataFrame with role annotations as the new song_features table
import sqlite3
import pandas as pd
conn = sqlite3.connect('dbs/song_features.db')

cols_to_keep = ['track_uri', 'track_name', 'artist_name', 'in_number_of_playlists', 'avg_playlist_followers', 'avg_position_in_playlist',
                 'NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist', 'cluster', 'role_annotation']
final_df = merged_df[cols_to_keep].copy()

final_df.to_sql('temp_song_features', conn, if_exists='replace', index=False)
cursor = conn.cursor()

try:
    cursor.execute(f"DROP TABLE IF EXISTS song_features")
    conn.commit()
    cursor.execute("ALTER TABLE temp_song_features RENAME TO song_features")
    conn.commit()
    print("successfully updated song_features table with role annotations")
except sqlite3.Error as e:
    print(f"Error updating table: {e}")

conn.close()

successfully updated song_features table with role annotations


In [6]:
conn = sqlite3.connect('dbs/song_features.db')
# Verify changes
query_result = pd.read_sql_query("SELECT * FROM song_features LIMIT 5", conn)
conn.close()
print(query_result)

  track_uri                   track_name       artist_name  \
0      None                      HUMBLE.    Kendrick Lamar   
1      None                    One Dance             Drake   
2      None  Broccoli (feat. Lil Yachty)              DRAM   
3      None                       Closer  The Chainsmokers   
4      None              Congratulations       Post Malone   

   in_number_of_playlists  avg_playlist_followers  avg_position_in_playlist  \
0                   45394                2.385668                 54.925431   
1                   41707                2.169924                 44.091112   
2                   40659                2.019110                 44.844315   
3                   40629                2.177090                 48.726033   
4                   39577                2.188721                 50.883089   

   NORM_in_number_of_playlists  NORM_avg_playlist_followers  \
0                   127.742867                    -0.023190   
1                   117.36